In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip


builder = (
    SparkSession.builder
    .appName("olist-bronze-ingestion-notebook")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint",       "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key",     "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key",     "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access","true")
    .config("spark.hadoop.fs.s3a.impl",           "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.sql.shuffle.partitions",       "4") 
    .config("spark.default.parallelism",          "4")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()


DATA_DIR = "/home/jovyan/data/"
BRONZE   = "s3a://bronze/csv/"

TABLES = {
    "orders": {"file": "olist_orders_dataset.csv"},
    "order_items": {"file": "olist_order_items_dataset.csv"},
    "order_payments": {"file": "olist_order_payments_dataset.csv"},
    "order_reviews": {"file": "olist_order_reviews_dataset.csv"},
    "customers": {"file": "olist_customers_dataset.csv"},
    "sellers": {"file": "olist_sellers_dataset.csv"},
    "products": {"file": "olist_products_dataset.csv"},
    "geolocation": {"file": "olist_geolocation_dataset.csv"},
    "category_translation": {"file": "product_category_name_translation.csv"},
}

for table_name, cfg in TABLES.items():
    print(f"Ingesting {table_name}...", end=" ")
    df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true") 
            .option("quote", "\"")         
            .option("escape", "\"")        
            .option("multiLine", "true")  
            .csv(DATA_DIR + cfg["file"])
    )
    df = df.withColumn("_ingested_at", F.current_timestamp()).withColumn("_source_file", F.lit(cfg["file"]))
    
    target_path = f"{BRONZE}{table_name}/"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(target_path)
    print("Done.")



Ingesting orders... Done.
Ingesting order_items... Done.
Ingesting order_payments... Done.
Ingesting order_reviews... Done.
Ingesting customers... Done.
Ingesting sellers... Done.
Ingesting products... Done.
Ingesting geolocation... Done.
Ingesting category_translation... Done.
